# Phase 3: Model Training (6 models)

This is where the models actually learn. We train six of them on the same training data, test them on the same test data, and record how accurate and how fast each one is.

| Model | Type | Why it's in the study |
|---|---|---|
| Linear Regression | classic statistics | the simplest possible model; anything fancier has to beat it |
| Decision Tree | single tree | easy to understand, very cheap |
| Random Forest | many trees, averaged | a popular accurate choice in cloud-workload papers, but heavy |
| XGBoost | boosted trees | strong on tabular data and widely used |
| LightGBM | boosted trees | designed to be fast and light |
| LSTM | deep learning (recurrent network) | tests whether deep learning's extra cost buys extra accuracy, which is the central question of this paper |

All models use **their library's default settings** here. Tuning comes in Phase 4, so this notebook gives the "out of the box" picture.

**What's measured for each model:**
- **MAE**: average error in CPU percentage points (lower is better)
- **RMSE**: like MAE, but big misses count more (lower is better)
- **MAPE**: average error as a percentage of the actual value. Only computed where actual CPU is at least 1%, see Phase 2 (lower is better)
- **R²**: share of the ups and downs in CPU the model explains; 1.0 would be perfect (higher is better)
- **Training time**: seconds to train
- **Inference time**: seconds to predict the whole test set (80,303 predictions)

**Reference row, not one of the 6 models:** we also score a "persistence" forecast that simply says *next CPU = current CPU*. It needs no training at all. If a model can't beat this, it isn't learning anything useful, so it's an important sanity check for the paper.

**What comes out:** trained models in `data/processed/models/`, test-set predictions (`test_predictions.parquet`, for the charts in Phase 10), the results table in `results/tables/phase3_results.*`, and the log `results/logs/phase03_model_training.log`.

**Run time on free Colab:** about 30–40 minutes. Random Forest takes about 13 minutes and the LSTM about 10–15; everything else takes seconds.

**If Colab disconnects:** just run everything again. Each model is saved as soon as it finishes, and finished models are skipped on the next run. To retrain everything from scratch, set `FORCE_RETRAIN = True` in the cell below.

## 0. Setup

In [1]:
# Mount Google Drive (Colab) or use the local Drive folder (PC), then load our shared code.
import os, sys, importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Green_ML_Project'
except ImportError:
    PROJECT_ROOT = r'H:\My Drive\Green_ML_Project'   # running on the PC instead of Colab

os.environ['GREEN_ML_ROOT'] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))

import config as C
import utils as U
importlib.reload(C); importlib.reload(U)   # picks up any edits to the scripts without restarting

print('Project folder:', C.PROJECT_ROOT)
print('Folder found:  ', os.path.isdir(C.PROJECT_ROOT))

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Green_ML_Project
Folder found:   True


In [2]:
import gc, json, os, platform, pickle
import numpy as np
import pandas as pd
import joblib
import model_definitions as M
importlib.reload(M)

FORCE_RETRAIN = False     # True = ignore saved models and train everything again

pd.set_option('display.width', 160)
log = U.get_logger('phase03_model_training')

import sklearn, xgboost, lightgbm, torch
log.info(f'scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__} | '
         f'lightgbm {lightgbm.__version__} | torch {torch.__version__}')
log.info(f'CPU cores: {os.cpu_count()} | GPU available: {torch.cuda.is_available()} (not used; all models run on CPU)')
log.info(f'Random seed: {C.RANDOM_SEED} | FORCE_RETRAIN = {FORCE_RETRAIN}')

RUN STARTED 2026-10-04 04:49:46 | phase03_model_training
python 3.13.15 | pandas 2.2.3 | numpy 2.1.3 | machine: Linux-6.6.122+-x86_64-with-glibc2.39
project root: /content/drive/MyDrive/Green_ML_Project
scikit-learn 1.6.1 | xgboost 3.4.1 | lightgbm 4.6.0 | torch 2.11.0+cpu
CPU cores: 2 | GPU available: False (not used; all models run on CPU)
Random seed: 42 | FORCE_RETRAIN = False


## 1. Load the data from Phase 2

In [3]:
train = pd.read_parquet(C.DATA_PROCESSED / 'train.parquet')
test = pd.read_parquet(C.DATA_PROCESSED / 'test.parquet')
features_main = pd.read_parquet(C.DATA_PROCESSED / 'features_main.parquet')
spec = json.load(open(C.DATA_PROCESSED / 'feature_columns.json'))
FEATURES, TARGET = spec['features'], spec['target']
assert FEATURES == U.FEATURE_COLUMNS, 'feature list changed since Phase 2 - rerun Phase 2'

X_train, y_train = train[FEATURES], train[TARGET].to_numpy()
X_test, y_test = test[FEATURES], test[TARGET].to_numpy()
log.info(f'Training rows: {len(train):,} | test rows: {len(test):,} | features: {len(FEATURES)}')

# Where each model's results are stored as soon as it finishes (this is what makes resuming work)
RESULTS_FILE = C.MODELS_DIR / 'phase3_results.json'
results = {} if FORCE_RETRAIN or not RESULTS_FILE.exists() else json.load(open(RESULTS_FILE))
if results:
    log.info(f'Already finished in an earlier run (will be skipped): {list(results)}')

def save_progress(name, row, predictions):
    results[name] = row
    json.dump(results, open(RESULTS_FILE, 'w'), indent=2)
    np.save(C.MODELS_DIR / f'pred_test_{M.slug(name)}.npy', predictions)

Training rows: 321,115 | test rows: 80,303 | features: 17


## 2. Reference: persistence forecast (no model)

"Next CPU = current CPU". It costs nothing to compute, and every real model should beat it.

In [4]:
persistence_pred = test['cpu_usage_pct'].to_numpy()
persistence = U.regression_metrics(y_test, persistence_pred)
log.info('Persistence reference: ' + ' | '.join(f'{k} {v:.4f}' for k, v in persistence.items()))

Persistence reference: MAE 0.7331 | RMSE 4.6575 | MAPE 13.9234 | R2 0.9035


## 3. The five tabular models

For each one: create it with default settings → train it on the training rows (timed) → predict the test rows (timed) → score the predictions → save the model to Drive.

Random Forest models are big (around 1–2 GB in memory), so we save them compressed. That makes saving slower, but the time is **not** counted as training time.

In [5]:
for name in M.TABULAR_MODELS:
    if name in results:
        log.info(f'{name}: already done, skipping')
        continue
    log.info(f'{name}: training on {len(X_train):,} rows ...')
    model = M.make_model(name, seed=C.RANDOM_SEED)
    with U.Timer() as t_fit:
        model.fit(X_train, y_train)
    with U.Timer() as t_pred:
        pred = model.predict(X_test)
    scores = U.regression_metrics(y_test, pred)

    path = C.MODELS_DIR / f'{M.slug(name)}.joblib'
    with U.Timer() as t_save:
        joblib.dump(model, path, compress=3 if name == 'Random Forest' else 0)
    row = {**scores, 'train_time_sec': t_fit.seconds, 'inference_time_sec': t_pred.seconds,
           'model_size_mb': path.stat().st_size / 1e6, 'params': {k: str(v) for k, v in model.get_params().items()}}
    save_progress(name, row, pred)
    log.info(f'{name}: MAE {scores["MAE"]:.4f} | RMSE {scores["RMSE"]:.4f} | MAPE {scores["MAPE"]:.2f}% | '
             f'R2 {scores["R2"]:.4f} | train {t_fit.seconds:.2f}s | predict {t_pred.seconds:.3f}s | '
             f'saved {row["model_size_mb"]:.1f} MB in {t_save.seconds:.1f}s')
    del model; gc.collect()

Linear Regression: training on 321,115 rows ...
Linear Regression: MAE 0.9032 | RMSE 4.5899 | MAPE 18.24% | R2 0.9062 | train 0.34s | predict 0.023s | saved 0.0 MB in 0.0s
Decision Tree: training on 321,115 rows ...
Decision Tree: MAE 0.8468 | RMSE 5.2473 | MAPE 15.20% | R2 0.8775 | train 11.46s | predict 0.042s | saved 24.9 MB in 0.1s
Random Forest: training on 321,115 rows ...
Random Forest: MAE 0.6903 | RMSE 3.6853 | MAPE 12.97% | R2 0.9396 | train 611.50s | predict 2.323s | saved 333.0 MB in 27.1s
XGBoost: training on 321,115 rows ...
XGBoost: MAE 0.7007 | RMSE 3.7376 | MAPE 13.92% | R2 0.9378 | train 3.18s | predict 0.164s | saved 0.5 MB in 0.2s
LightGBM: training on 321,115 rows ...
LightGBM: MAE 0.6743 | RMSE 3.7123 | MAPE 12.74% | R2 0.9387 | train 2.32s | predict 0.165s | saved 0.3 MB in 0.0s


## 4. The LSTM

The LSTM doesn't read one row at a time like the other models. It reads a **short history**: the last 12 readings (one hour) of CPU% and memory%, in order, and predicts the next CPU%. We build these windows from the same Phase 2 data, so:
- the LSTM predicts **exactly the same test rows** as the other models, which keeps the comparison fair
- a window for an early test row may reach back into the training period. That's fine, because it's only past data, which a real deployment would also have.

Settings (agreed by the team):

| Setting | Value |
|---|---|
| Layers | 1 LSTM layer (64 units) → 1 output |
| Inputs per step | CPU% and memory% (divided by 100) |
| History length | 12 readings = 1 hour |
| Optimizer | Adam, learning rate 0.001, batch 256 |
| Epochs | at most 15, stopping early if it doesn't improve for 3 epochs in a row |

**Early stopping** needs data the LSTM doesn't train on, so we hold back the last 10% of each VM's *training* period as a validation slice. The test data is never touched during training. As a result the LSTM trains on about 90% of the rows the tree models get. That's the standard way to train neural networks, and the paper will say so.

Training time includes every epoch, including the ones after which it stopped improving.

In [6]:
name = 'LSTM'
if name in results:
    log.info('LSTM: already done, skipping')
else:
    p = M.LSTM_DEFAULTS
    with U.Timer() as t_prep:
        ds = M.lstm_datasets(features_main, p['seq_len'])
    (Xtr, ytr), (Xva, yva), (Xte, yte, te_idx) = ds['train'], ds['val'], ds['test']
    # The LSTM test windows must line up one-to-one with test.parquet
    assert len(Xte) == len(test) and np.allclose(yte * 100, y_test, atol=1e-3), 'LSTM test rows do not match'
    log.info(f'LSTM windows: train {len(Xtr):,} | validation {len(Xva):,} | test {len(Xte):,} '
             f'(built in {t_prep.seconds:.1f}s, shape {Xtr.shape[1:]})')

    with U.Timer() as t_fit:
        model, history = M.train_lstm(Xtr, ytr, Xva, yva, seed=C.RANDOM_SEED, log_fn=log.info)
    with U.Timer() as t_pred:
        pred = M.predict_lstm(model, Xte)
    scores = U.regression_metrics(y_test, pred)

    path = C.MODELS_DIR / 'lstm.pt'
    torch.save(model.state_dict(), path)
    best_epoch = int(np.nanargmin([h['val_loss'] for h in history]) + 1)
    row = {**scores, 'train_time_sec': t_fit.seconds, 'inference_time_sec': t_pred.seconds,
           'model_size_mb': path.stat().st_size / 1e6, 'epochs_run': len(history), 'best_epoch': best_epoch,
           'params': {k: str(v) for k, v in p.items()}}
    pd.DataFrame(history).to_csv(C.MODELS_DIR / 'lstm_training_history.csv', index=False)
    save_progress(name, row, pred)
    log.info(f'LSTM: MAE {scores["MAE"]:.4f} | RMSE {scores["RMSE"]:.4f} | MAPE {scores["MAPE"]:.2f}% | '
             f'R2 {scores["R2"]:.4f} | train {t_fit.seconds:.1f}s ({len(history)} epochs, best {best_epoch}) | '
             f'predict {t_pred.seconds:.2f}s')
    del model; gc.collect()

LSTM windows: train 288,436 | validation 32,129 | test 80,303 (built in 0.3s, shape (12, 2))
  epoch  1: train_mse=0.00338 val_mse=0.00095 (24.9s)
  epoch  2: train_mse=0.00163 val_mse=0.00079 (26.7s)
  epoch  3: train_mse=0.00161 val_mse=0.00084 (26.7s)
  epoch  4: train_mse=0.00159 val_mse=0.00081 (26.7s)
  epoch  5: train_mse=0.00156 val_mse=0.00078 (27.2s)
  epoch  6: train_mse=0.00154 val_mse=0.00078 (23.6s)
  epoch  7: train_mse=0.00151 val_mse=0.00079 (26.8s)
  epoch  8: train_mse=0.00149 val_mse=0.00079 (27.0s)
  early stop at epoch 8 (best val_mse=0.00078)
LSTM: MAE 0.8774 | RMSE 4.5345 | MAPE 16.61% | R2 0.9085 | train 217.5s (8 epochs, best 5) | predict 2.22s


## 5. Results

In [7]:
order = M.TABULAR_MODELS + ['LSTM']
res = pd.DataFrame([{'Model': n, **{k: results[n][k] for k in
        ['MAE', 'RMSE', 'MAPE', 'R2', 'train_time_sec', 'inference_time_sec', 'model_size_mb']}} for n in order])
res['inference_ms_per_1k'] = 1000 * res['inference_time_sec'] / (len(test) / 1000)
ref = pd.DataFrame([{'Model': 'Persistence (reference)', **persistence, 'train_time_sec': 0.0,
                     'inference_time_sec': 0.0, 'model_size_mb': 0.0, 'inference_ms_per_1k': 0.0}])
table = pd.concat([res, ref], ignore_index=True)

U.log_df(log, table.round(4), 'PHASE 3 RESULTS (default settings, test set)')
show = table.rename(columns={'MAPE': 'MAPE (%)', 'train_time_sec': 'Train time (s)',
                             'inference_time_sec': 'Inference time (s)', 'model_size_mb': 'Size (MB)',
                             'inference_ms_per_1k': 'Inference (ms / 1k rows)'})
U.save_table(show, 'phase3_results', 'Phase 3: default models on the test set', float_fmt='{:.4f}')
table.to_csv(C.TABLES_DIR / 'phase3_results_raw.csv', index=False)
show.round(4)

--- PHASE 3 RESULTS (default settings, test set) ---
                     Model     MAE    RMSE     MAPE      R2  train_time_sec  inference_time_sec  model_size_mb  inference_ms_per_1k
0        Linear Regression  0.9032  4.5899  18.2409  0.9062          0.3352              0.0230         0.0013               0.2859
1            Decision Tree  0.8468  5.2473  15.1961  0.8775         11.4586              0.0417        24.9369               0.5187
2            Random Forest  0.6903  3.6853  12.9664  0.9396        611.5020              2.3232       332.9655              28.9305
3                  XGBoost  0.7007  3.7376  13.9168  0.9378          3.1828              0.1638         0.4668               2.0401
4                 LightGBM  0.6743  3.7123  12.7388  0.9387          2.3208              0.1647         0.2828               2.0505
5                     LSTM  0.8774  4.5345  16.6097  0.9085        217.5143              2.2223         0.0726              27.6735
6  Persistence (referen

,Model,MAE,RMSE,MAPE (%),R2,Train time (s),Inference time (s),Size (MB),Inference (ms / 1k rows)
0,Linear Regression,0.9032,4.5899,18.2409,0.9062,0.3352,0.0230,0.0013,0.2859
1,Decision Tree,0.8468,5.2473,15.1961,0.8775,11.4586,0.0417,24.9369,0.5187
2,Random Forest,0.6903,3.6853,12.9664,0.9396,611.5020,2.3232,332.9655,28.9305
3,XGBoost,0.7007,3.7376,13.9168,0.9378,3.1828,0.1638,0.4668,2.0401
4,LightGBM,0.6743,3.7123,12.7388,0.9387,2.3208,0.1647,0.2828,2.0505
5,LSTM,0.8774,4.5345,16.6097,0.9085,217.5143,2.2223,0.0726,27.6735
6,Persistence (reference),0.7331,4.6575,13.9234,0.9035,0.0000,0.0000,0.0000,0.0000


In [8]:
# Who wins each measure, and does every model beat doing nothing (persistence)?
best = {'Lowest MAE': res.loc[res.MAE.idxmin(), 'Model'], 'Lowest RMSE': res.loc[res.RMSE.idxmin(), 'Model'],
        'Lowest MAPE': res.loc[res.MAPE.idxmin(), 'Model'], 'Highest R2': res.loc[res.R2.idxmax(), 'Model'],
        'Fastest training': res.loc[res.train_time_sec.idxmin(), 'Model'],
        'Fastest inference': res.loc[res.inference_time_sec.idxmin(), 'Model']}
for k, v in best.items():
    log.info(f'{k:<18}: {v}')
for _, r in res.iterrows():
    log.info(f'{r.Model:<17} beats persistence on MAE: {r.MAE < persistence["MAE"]} | on RMSE: {r.RMSE < persistence["RMSE"]}')

Lowest MAE        : LightGBM
Lowest RMSE       : Random Forest
Lowest MAPE       : LightGBM
Highest R2        : Random Forest
Fastest training  : Linear Regression
Fastest inference : Linear Regression
Linear Regression beats persistence on MAE: False | on RMSE: True
Decision Tree     beats persistence on MAE: False | on RMSE: False
Random Forest     beats persistence on MAE: True | on RMSE: True
XGBoost           beats persistence on MAE: True | on RMSE: True
LightGBM          beats persistence on MAE: True | on RMSE: True
LSTM              beats persistence on MAE: False | on RMSE: True


In [9]:
# Save every model's test predictions side by side (used for the actual-vs-predicted chart in Phase 10)
preds = test[['vm_id', 'timestamp', TARGET]].rename(columns={TARGET: 'actual'}).copy()
preds['persistence'] = persistence_pred
for n in order:
    preds[M.slug(n)] = np.load(C.MODELS_DIR / f'pred_test_{M.slug(n)}.npy')
preds.to_parquet(C.DATA_PROCESSED / 'test_predictions.parquet', index=False)

for f in sorted(C.MODELS_DIR.iterdir()):
    log.info(f'Saved: data/processed/models/{f.name} ({f.stat().st_size / 1e6:.1f} MB)')
log.info('Saved: data/processed/test_predictions.parquet')
log.info('PHASE 3 COMPLETE')

Saved: data/processed/models/decision_tree.joblib (24.9 MB)
Saved: data/processed/models/lightgbm.joblib (0.3 MB)
Saved: data/processed/models/linear_regression.joblib (0.0 MB)
Saved: data/processed/models/lstm.pt (0.1 MB)
Saved: data/processed/models/lstm_training_history.csv (0.0 MB)
Saved: data/processed/models/phase3_results.json (0.0 MB)
Saved: data/processed/models/pred_test_decision_tree.npy (0.6 MB)
Saved: data/processed/models/pred_test_lightgbm.npy (0.6 MB)
Saved: data/processed/models/pred_test_linear_regression.npy (0.6 MB)
Saved: data/processed/models/pred_test_lstm.npy (0.3 MB)
Saved: data/processed/models/pred_test_random_forest.npy (0.6 MB)
Saved: data/processed/models/pred_test_xgboost.npy (0.3 MB)
Saved: data/processed/models/random_forest.joblib (333.0 MB)
Saved: data/processed/models/xgboost.joblib (0.5 MB)
Saved: data/processed/test_predictions.parquet
PHASE 3 COMPLETE


## Done: what to check

- Every model should have **R² well above 0** and should **beat the persistence reference** on MAE or RMSE (see the lines just above). If one doesn't, tell the team before moving on.
- Compare **training time** with **accuracy**. If a slow model is only slightly more accurate than a fast one, that's exactly the trade-off the TOPSIS ranking in Phase 7 is designed to weigh.

Next: `04_hyperparameter_tuning.ipynb`.